# Delta Table Internals Demo

This notebook shows the internal details of how Delta Tables are stored and handled under the hood.

In case of managed Databricks tables, the storage location is not exposed, therefore this demo uses an external delta table (from AWS S3). That's why the connection to an S3 bucket is a prerequisite for running this notebook. Follow the steps from [here](https://docs.databricks.com/aws/en/connect/unity-catalog/cloud-storage/s3/s3-external-location-manual) to set up an S3 connection from Databricks and don't forget to modify the S3 bucket paths to your own.

### Create managed delta table

In this case the storage location is not exposed (managed by Unity catalog).

In [0]:
%sql
CREATE TABLE IF NOT EXISTS employee (
  emp_id INT,
  emp_name STRING,
  dob DATE
)

In [0]:
%sql
INSERT INTO employee VALUES (1, 'John Doe', '1980-01-01');

In [0]:
%sql
DESCRIBE DETAIL employee

In [0]:
%sql
DESCRIBE HISTORY workspace.default.employee;

### Create external delta table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS employee_ext  (
  emp_id INT,
  emp_name STRING,
  dob DATE
)
USING DELTA
LOCATION 's3://szk-databricks-training-demo-s3/employee_ext';

In [0]:
%sql
DESCRIBE DETAIL employee_ext

In [0]:
%python
# List all files and folders from the base directory of the (delta) table
files_and_folders = dbutils.fs.ls("s3://szk-databricks-training-demo-s3/employee_ext")
for item in files_and_folders:
    print(item.name)

In [0]:
# List all files and folders from the _delta_log directory
files_and_folders = dbutils.fs.ls("s3://szk-databricks-training-demo-s3/employee_ext/_delta_log")
for item in files_and_folders:
    print(item.name)

In [0]:
# Display the contents of the log files
# Inspect the CRC file: contains table related metadata, table statistics
dbutils.fs.head("s3://szk-databricks-training-demo-s3/employee_ext/_delta_log/00000000000000000000.crc")

In [0]:
# Inspect the JSON files: more transaction related information, such as who performed the transaction, what operation was done (CREATE TABLE), notebook and cluser ids, etc.
dbutils.fs.head("s3://szk-databricks-training-demo-s3/employee_ext/_delta_log/00000000000000000000.json")

In [0]:
%sql
INSERT INTO employee_ext (emp_id, emp_name, dob)
VALUES (1, 'John', '1990-01-01')

In [0]:
%python
# List all files and folders from the base directory of the (delta) table
files_and_folders = dbutils.fs.ls("s3://szk-databricks-training-demo-s3/employee_ext")
for item in files_and_folders:
    print(item.name)

# ----> Note the data file that is created -> parquet file

In [0]:
# List all files and folders from the _delta_log directory
files_and_folders = dbutils.fs.ls("s3://szk-databricks-training-demo-s3/employee_ext/_delta_log")
for item in files_and_folders:
    print(item.name)

# ----> and the new JSON and CRC files

In [0]:
# Inspect the data file: contains the actual data in a compressed, columnar format
dbutils.fs.head("s3://szk-databricks-training-demo-s3/employee_ext/part-00000-cc254f94-986e-4657-a8bc-b2771a27d894.c000.snappy.parquet")

In [0]:
# Updated table level statistics: e.g. minValue, maxValue in the CRC file
dbutils.fs.head("s3://szk-databricks-training-demo-s3/employee_ext/_delta_log/00000000000000000001.crc")

In [0]:
# Updated transaction level details (e.g. "add" + path to new parquet file) in the JSON file
dbutils.fs.head("s3://szk-databricks-training-demo-s3/employee_ext/_delta_log/00000000000000000001.json")

In [0]:
%sql
INSERT INTO employee_ext (emp_id, emp_name, dob)
VALUES
(2, 'Jane', '1996-01-01'),
(3, 'John', '1995-01-15'),
(4, 'Sarah', '1994-05-01'),
(5, 'Ben', '1999-07-01')
    

In [0]:
# New parquet file added with the newly inserted data
files_and_folders = dbutils.fs.ls("s3://szk-databricks-training-demo-s3/employee_ext")
for item in files_and_folders:
    print(item.name)

In [0]:
# And new CRC + JSON files
files_and_folders = dbutils.fs.ls("s3://szk-databricks-training-demo-s3/employee_ext/_delta_log")
for item in files_and_folders:
    print(item.name)

In [0]:
%sql
UPDATE employee_ext
SET emp_name = "Monica"
WHERE emp_id = 3

In [0]:
# The update does not modify any existing data files, but creates a new one (for time travel)
files_and_folders = dbutils.fs.ls("s3://szk-databricks-training-demo-s3/employee_ext")
for item in files_and_folders:
    print(item.name)

In [0]:
# And new CRC + JSON files
files_and_folders = dbutils.fs.ls("s3://szk-databricks-training-demo-s3/employee_ext/_delta_log")
for item in files_and_folders:
    print(item.name)

In [0]:
# Marks the relevant datafile as removed and adds a new datafile with the updated column
dbutils.fs.head("s3://szk-databricks-training-demo-s3/employee_ext/_delta_log/00000000000000000004.json")

In [0]:
# Marks the relevant datafile as removed and adds a new datafile with the updated column
dbutils.fs.head("s3://szk-databricks-training-demo-s3/employee_ext/part-00000-d23dff80-856a-4b82-815b-4c3beba3b475.c000.snappy.parquet")

In [0]:
%sql
DELETE FROM employee_ext
WHERE emp_id = 2

In [0]:
# The delete file does not modify any existing data files, but creates a new one (for time travel)
files_and_folders = dbutils.fs.ls("s3://szk-databricks-training-demo-s3/employee_ext")
for item in files_and_folders:
    print(item.name)

In [0]:
# And new CRC + JSON files
files_and_folders = dbutils.fs.ls("s3://szk-databricks-training-demo-s3/employee_ext/_delta_log")
for item in files_and_folders:
    print(item.name)

In [0]:
# Note the "remove" entry with "numRecords: 5" and the "add" with "numRecords: 4"
dbutils.fs.head("s3://szk-databricks-training-demo-s3/employee_ext/_delta_log/00000000000000000006.json")